In [1]:
import os
import sys


os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Read Inside Airbnb data") \
    .getOrCreate()


In [2]:
listings = spark.read.csv("data/listings.csv.gz",
                          header= True,
                          inferSchema=True,
                          sep=",", #SEPARATOR
                          quote='"', #pentru valorile string
                          escape='"',
                          multiLine=True, 
                          mode="PERMISSIVE"
                         )

In [3]:
listings.printSchema()

root
 |-- id: long (nullable = true)
 |-- listing_url: string (nullable = true)
 |-- scrape_id: long (nullable = true)
 |-- last_scraped: date (nullable = true)
 |-- source: string (nullable = true)
 |-- name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- neighborhood_overview: string (nullable = true)
 |-- picture_url: string (nullable = true)
 |-- host_id: integer (nullable = true)
 |-- host_url: string (nullable = true)
 |-- host_name: string (nullable = true)
 |-- host_since: date (nullable = true)
 |-- host_location: string (nullable = true)
 |-- host_about: string (nullable = true)
 |-- host_response_time: string (nullable = true)
 |-- host_response_rate: string (nullable = true)
 |-- host_acceptance_rate: string (nullable = true)
 |-- host_is_superhost: string (nullable = true)
 |-- host_thumbnail_url: string (nullable = true)
 |-- host_picture_url: string (nullable = true)
 |-- host_neighbourhood: string (nullable = true)
 |-- host_listings_count: int

In [41]:
listings \
    .filter(listings.picture_url.isNotNull()
) \
.select('picture_url') \
.limit(1) \
.show(1, truncate = False)

+------------------------------------------------------------------------------------------------------+
|picture_url                                                                                           |
+------------------------------------------------------------------------------------------------------+
|https://a0.muscache.com/pictures/miso/Hosting-13913/original/d755aa6d-cebb-4464-80be-2722c921e8d5.jpeg|
+------------------------------------------------------------------------------------------------------+



In [5]:
viewed_listings = listings \
    .filter(listings.reviews_per_month > 10) \
    .count()

print(f"Numarul de proprietati cu peste 10 reviews pe lune este:{viewed_listings}")

Numarul de proprietati cu peste 10 reviews pe lune este:66


In [6]:
listings \
    .filter(listings.bathrooms > listings.bedrooms) \
    .select('name') \
    .show(1, truncate =  False)

+--------------------------------------------+
|name                                        |
+--------------------------------------------+
|Cosy Double studio in Zone 2 Hammersmith (1)|
+--------------------------------------------+
only showing top 1 row


In [15]:
from pyspark.sql.functions import regexp_replace

price_num_df = listings \
    .withColumn('price_num', regexp_replace('price', '[$,]', '') .cast('float')) \

price_num_df.schema['price_num']

StructField('price_num', FloatType(), True)

In [8]:
price_num_df.filter(price_num_df.price_num > 5000) \
    .select('name', 'price') \
    .collect()

[Row(name='Room in a cosy flat. Central, clean', price='$8,000.00'),
 Row(name='Spacious Private Ground Floor Room', price='$6,309.00'),
 Row(name='No Longer Available', price='$53,588.00'),
 Row(name='Bright & airy DoubleBed with EnSuite in Zone 2!', price='$74,100.00'),
 Row(name='The Apartments by The Sloane Club, Two Bedroom Apt', price='$7,377.00'),
 Row(name='The Apartments by The Sloane Club, L 2 Bedroom Apt', price='$7,377.00'),
 Row(name='Single room. 7ft x 9ft - Over looking garden', price='$6,523.00'),
 Row(name='Close To London Eye (TUR)', price='$6,666.00'),
 Row(name='Beautiful 2 BR flat in Kilburn with free parking', price='$6,000.00'),
 Row(name='Semi-detached mews house in Knightsbridge.', price='$7,019.00'),
 Row(name='Affordable Spacious  Room on the edge of the city', price='$6,000.00'),
 Row(name='Henry’s Townhouse, London', price='$6,500.00'),
 Row(name='City Suite', price='$5,353.00'),
 Row(name='Hyde Park Suite', price='$5,653.00'),
 Row(name='SHORT WALK TO LOND

In [19]:
price_num_df \
    .filter((price_num_df.price_num < 150) & (listings.number_of_reviews >20) &(listings.review_scores_rating > 4.5)) \
    .select('name', 'price', 'number_of_reviews', 'review_scores_rating') \
    .show(truncate = False)

+-------------------------------------------------+-------+-----------------+--------------------+
|name                                             |price  |number_of_reviews|review_scores_rating|
+-------------------------------------------------+-------+-----------------+--------------------+
|Holiday London DB Room Let-on going              |$70.00 |55               |4.85                |
|Bright Chelsea  Apartment. Chelsea!              |$149.00|97               |4.8                 |
|You are GUARANTEED to love this                  |$90.00 |730              |4.87                |
|SUNNY ROOM PRIVATE BATHROOM PLUS BREAKFAST       |$61.00 |387              |4.77                |
|SPACIOUS ROOM IN CONTEMPORARY STYLE FLAT         |$49.00 |72               |4.97                |
|Room with a view, shared flat,  central  Bankside|$96.00 |137              |4.7                 |
|You Will Save Money Here                         |$71.00 |639              |4.89                |
|Quiet Com

In [20]:
price_num_df \
    .filter("price_num <150 OR bathrooms>1") \
    .select('name') \
    .show(truncate = False)

+--------------------------------------------------+
|name                                              |
+--------------------------------------------------+
|Holiday London DB Room Let-on going               |
|Bright Chelsea  Apartment. Chelsea!               |
|Very Central Modern 3-Bed/2 Bath By Oxford St W1  |
|Kew Gardens 3BR house in cul-de-sac               |
|You are GUARANTEED to love this                   |
|SUNNY ROOM PRIVATE BATHROOM PLUS BREAKFAST        |
|Short Term Home                                   |
|SPACIOUS ROOM IN CONTEMPORARY STYLE FLAT          |
|Room with a view, shared flat,  central  Bankside |
|You Will Save Money Here                          |
|Quiet Comfortable Room in Fulham                  |
|Room with a garden                                |
|Pleasant Single Room in zone 1.                   |
|Cosy Double studio in Zone 2 Hammersmith (6)      |
|Beautiful Small Studio Hammersmith                |
|Close to Wimbledon All England Tennis -huge d

In [42]:
from pyspark.sql.functions import max

max_row = price_num_df.select(max('price_num')).collect()
max_value = max_row[0][0] 

print(f"Most expensive listing is: {max_value}")


price_num_df \
    .filter(price_num_df.price_num == max_value) \
    .select('name', 'price_num') \
    .show(truncate=False)


##OR
#price_num_df \
#    .select(max('price_num')) \
#    .show()

+--------------+
|max(price_num)|
+--------------+
|     1085147.0|
+--------------+



In [43]:
from pyspark.sql.functions import desc

price_num_df \
    .orderBy(desc('reviews_per_month')) \
    .select('name', 'reviews_per_month', 'price_num') \
    .show(1, truncate = False)

+--------------------+-----------------+---------+
|name                |reviews_per_month|price_num|
+--------------------+-----------------+---------+
|Double Room+ Ensuite|36.96            |162.0    |
+--------------------+-----------------+---------+
only showing top 1 row


In [37]:
from pyspark.sql.functions import count

price_num_df \
    .select('host_id') \
    .distinct() \
    .count() 

55646

In [45]:
from pyspark.sql.functions import year

first_review_2024 = price_num_df \
    .filter(year('first_review') == 2024) \
    .select('name', 'first_review')

first_review_2024.show(10, truncate = False)

+--------------------------------------------------+------------+
|name                                              |first_review|
+--------------------------------------------------+------------+
|Close to Wimbledon All England Tennis -huge double|2024-08-11  |
|one Double bed room with en-suite facilities      |2024-03-21  |
|Bridgerton inspired cottage core apartment        |2024-09-14  |
|Sm double room  with own bathroom                 |2024-06-04  |
|Central, modern pied-a-terre                      |2024-11-29  |
|Superlux flat in Knightsbridge                    |2024-01-01  |
|The Pink House, Notting Hill                      |2024-07-14  |
|Stylish garden flat in Hackney                    |2024-09-15  |
|Luxurious Flat in South Kensington                |2024-06-19  |
|Double Standard Room (Ensuite)                    |2024-09-01  |
+--------------------------------------------------+------------+
only showing top 10 rows
